<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/CCCustomerLTV/clv_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Customer Data Generation

This notebook demonstrates how to generate synthetic credit card customer data for Customer Lifetime Value (CLV) analysis.

## 📋 Learning Objectives
- Understand the components of credit card customer data
- Learn to generate realistic synthetic financial data
- Explore data quality and validation techniques
- Prepare data for CLV analysis

## 🏦 Business Context
Banks need comprehensive customer data to calculate CLV accurately. This includes:
- Customer demographics and behavior
- Transaction patterns and payment history
- Revenue and cost components
- Risk indicators and profitability metrics

## 1. Setup and Dependencies

In [ ]:
# Install required packages (run this cell in Google Colab)
!pip install pandas numpy matplotlib seaborn faker scikit-learn

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
from faker.providers import BaseProvider
import random
from datetime import datetime, timedelta
import warnings
import os

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8')
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")

## 2. Custom Provider for Credit Card Data

In [ ]:
class CreditCardProvider(BaseProvider):
    """Custom provider for credit card specific data"""
    
    card_types = ['Gold', 'Platinum', 'Silver', 'Premium', 'Classic']
    card_networks = ['Visa', 'Mastercard', 'RuPay']
    employment_types = ['Salaried', 'Self-Employed', 'Business Owner', 'Professional', 'Retired']
    cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Pune', 'Ahmedabad', 
              'Kolkata', 'Jaipur', 'Lucknow', 'Indore', 'Nagpur', 'Surat', 'Vadodara']
    
    def card_type(self):
        return self.random_element(self.card_types)
    
    def card_network(self):
        return self.random_element(self.card_networks)
    
    def employment_type(self):
        return self.random_element(self.employment_types)
    
    def indian_city(self):
        return self.random_element(self.cities)

print("✅ Custom provider created successfully!")

## 3. Data Generation Configuration

In [ ]:
# Configuration parameters - Feel free to modify these!
CONFIG = {
    'num_records': 10000,
    'seed': 42,
    'min_age': 18,
    'max_age': 75,
    'min_income': 200000,
    'max_income': 2000000,
    'min_tenure': 1,
    'max_tenure': 120,
    'churn_rate': 0.15
}

print("📊 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

## 4. Generate Synthetic Data

In [ ]:
def generate_synthetic_data(num_records=10000, seed=42, **kwargs):
    """Generate synthetic credit card customer data"""
    
    # Set random seeds
    random.seed(seed)
    np.random.seed(seed)
    
    # Initialize Faker
    fake = Faker('en_IN')
    fake.add_provider(CreditCardProvider)
    Faker.seed(seed)
    
    # Extract parameters
    min_age = kwargs.get('min_age', 18)
    max_age = kwargs.get('max_age', 75)
    min_income = kwargs.get('min_income', 200000)
    max_income = kwargs.get('max_income', 2000000)
    min_tenure = kwargs.get('min_tenure', 1)
    max_tenure = kwargs.get('max_tenure', 120)
    churn_rate = kwargs.get('churn_rate', 0.15)
    
    data = []
    
    print(f"Generating {num_records:,} synthetic records...")
    
    for i in range(num_records):
        if (i + 1) % 1000 == 0:
            print(f"Generated {i + 1:,} records...")
        
        # Basic demographics
        age = random.randint(min_age, max_age)
        gender = random.choice(['Male', 'Female'])
        city = fake.indian_city()
        annual_income = random.randint(min_income, max_income)
        employment_type = fake.employment_type()
        
        # Account information
        tenure_months = random.randint(min_tenure, max_tenure)
        card_type = fake.card_type()
        card_network = fake.card_network()
        
        # Credit limit based on income and profile
        income_multiplier = random.uniform(0.3, 2.0)
        if card_type in ['Platinum', 'Premium']:
            income_multiplier *= random.uniform(1.2, 2.5)
        credit_limit = min(int(annual_income * income_multiplier / 12), 500000)
        
        # Usage patterns
        avg_monthly_spend = random.uniform(0.1, 0.8) * credit_limit
        utilization_rate = avg_monthly_spend / credit_limit
        transactions_per_month = max(1, int(np.random.poisson(15) * (avg_monthly_spend / 10000)))
        
        # Payment behavior
        payment_ratio = random.uniform(0.05, 1.0)
        if payment_ratio < 0.1:
            payment_ratio = random.uniform(0.05, 0.15)
        
        late_payments_12m = max(0, int(np.random.poisson(1.5) * (1 - payment_ratio)))
        
        # Current balance
        current_balance = avg_monthly_spend * random.uniform(0.5, 2.0)
        current_balance = min(current_balance, credit_limit)
        
        # Customer status
        is_active = random.random() > churn_rate
        
        # Revenue components
        interchange_revenue = avg_monthly_spend * random.uniform(0.015, 0.025)
        interest_revenue = current_balance * random.uniform(0.015, 0.035) if payment_ratio < 0.9 else 0
        annual_fee = random.choice([0, 500, 1500, 3000, 5000]) if card_type != 'Classic' else 0
        monthly_fee_revenue = annual_fee / 12
        
        total_monthly_revenue = interchange_revenue + interest_revenue + monthly_fee_revenue
        
        # Costs
        acquisition_cost = random.uniform(2000, 8000) / tenure_months
        servicing_cost = random.uniform(100, 300)
        default_risk_cost = current_balance * random.uniform(0.001, 0.01) * (late_payments_12m / 12)
        
        total_monthly_cost = acquisition_cost + servicing_cost + default_risk_cost
        monthly_profit = total_monthly_revenue - total_monthly_cost
        
        record = {
            'customer_id': f'CC_{i+1:06d}',
            'age': age,
            'gender': gender,
            'city': city,
            'annual_income': annual_income,
            'employment_type': employment_type,
            'tenure_months': tenure_months,
            'card_type': card_type,
            'card_network': card_network,
            'credit_limit': int(credit_limit),
            'current_balance': round(current_balance, 2),
            'utilization_rate': round(utilization_rate, 3),
            'avg_monthly_spend': round(avg_monthly_spend, 2),
            'transactions_per_month': transactions_per_month,
            'payment_ratio': round(payment_ratio, 3),
            'late_payments_12m': late_payments_12m,
            'is_active': is_active,
            'interchange_revenue': round(interchange_revenue, 2),
            'interest_revenue': round(interest_revenue, 2),
            'annual_fee': annual_fee,
            'monthly_revenue': round(total_monthly_revenue, 2),
            'monthly_cost': round(total_monthly_cost, 2),
            'monthly_profit': round(monthly_profit, 2),
            'last_transaction_date': fake.date_between(start_date='-3M', end_date='today'),
            'account_opening_date': fake.date_between(start_date=f'-{tenure_months+12}M', end_date=f'-{tenure_months}M')
        }
        
        data.append(record)
    
    df = pd.DataFrame(data)
    print(f"✅ Generated {len(df):,} records successfully!")
    return df

# Generate the data
df = generate_synthetic_data(**CONFIG)

## 5. Data Exploration and Validation

In [ ]:
# Basic information about the dataset
print("📊 DATASET OVERVIEW")
print("=" * 50)
print(f"Number of records: {len(df):,}")
print(f"Number of features: {len(df.columns)}")
print(f"Dataset size: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n📋 COLUMN INFORMATION")
print("=" * 50)
print(df.info())

In [ ]:
# Display first few records
print("🔍 SAMPLE RECORDS")
print("=" * 50)
display(df.head())

In [ ]:
# Statistical summary
print("📈 STATISTICAL SUMMARY")
print("=" * 50)
display(df.describe())

## 6. Data Quality Checks

In [ ]:
# Check for missing values
print("🔍 DATA QUALITY CHECKS")
print("=" * 50)

missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("✅ No missing values found!")
else:
    print("⚠️ Missing values found:")
    print(missing_values[missing_values > 0])

# Check for duplicates
duplicates = df.duplicated().sum()
if duplicates == 0:
    print("✅ No duplicate records found!")
else:
    print(f"⚠️ Found {duplicates} duplicate records")

# Validate data ranges
print("\n📊 DATA RANGE VALIDATION")
print("-" * 30)
validations = [
    ("Age range", df['age'].min(), df['age'].max(), "18-75"),
    ("Income range", f"₹{df['annual_income'].min():,}", f"₹{df['annual_income'].max():,}", "₹200K-₹2M"),
    ("Utilization rate", f"{df['utilization_rate'].min():.3f}", f"{df['utilization_rate'].max():.3f}", "0-1"),
    ("Payment ratio", f"{df['payment_ratio'].min():.3f}", f"{df['payment_ratio'].max():.3f}", "0-1")
]

for metric, min_val, max_val, expected in validations:
    print(f"{metric}: {min_val} - {max_val} (Expected: {expected})")

## 7. Data Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Synthetic Credit Card Customer Data Overview', fontsize=16, fontweight='bold')

# Age distribution
axes[0,0].hist(df['age'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')
axes[0,0].set_ylabel('Count')

# Income distribution
axes[0,1].hist(df['annual_income']/100000, bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0,1].set_title('Annual Income Distribution')
axes[0,1].set_xlabel('Income (Lakhs ₹)')
axes[0,1].set_ylabel('Count')

# Monthly profit distribution
axes[0,2].hist(df['monthly_profit'], bins=30, alpha=0.7, color='lightcoral', edgecolor='black')
axes[0,2].set_title('Monthly Profit Distribution')
axes[0,2].set_xlabel('Monthly Profit (₹)')
axes[0,2].set_ylabel('Count')
axes[0,2].axvline(0, color='red', linestyle='--', label='Break-even')
axes[0,2].legend()

# Card type distribution
card_counts = df['card_type'].value_counts()
axes[1,0].pie(card_counts.values, labels=card_counts.index, autopct='%1.1f%%')
axes[1,0].set_title('Card Type Distribution')

# Utilization rate vs Monthly spend
scatter = axes[1,1].scatter(df['utilization_rate'], df['avg_monthly_spend'], 
                           alpha=0.6, c=df['monthly_profit'], cmap='RdYlBu_r')
axes[1,1].set_title('Utilization Rate vs Monthly Spend')
axes[1,1].set_xlabel('Utilization Rate')
axes[1,1].set_ylabel('Average Monthly Spend (₹)')
plt.colorbar(scatter, ax=axes[1,1], label='Monthly Profit')

# Tenure distribution
axes[1,2].hist(df['tenure_months'], bins=30, alpha=0.7, color='orange', edgecolor='black')
axes[1,2].set_title('Customer Tenure Distribution')
axes[1,2].set_xlabel('Tenure (Months)')
axes[1,2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 8. Business Insights from Synthetic Data

In [ ]:
# Business metrics analysis
print("💼 BUSINESS INSIGHTS")
print("=" * 50)

# Profitability analysis
profitable_customers = df[df['monthly_profit'] > 0]
unprofitable_customers = df[df['monthly_profit'] <= 0]

print(f"📊 Customer Profitability:")
print(f"  • Profitable customers: {len(profitable_customers):,} ({len(profitable_customers)/len(df)*100:.1f}%)")
print(f"  • Unprofitable customers: {len(unprofitable_customers):,} ({len(unprofitable_customers)/len(df)*100:.1f}%)")
print(f"  • Average profit (profitable): ₹{profitable_customers['monthly_profit'].mean():,.2f}")
print(f"  • Average loss (unprofitable): ₹{unprofitable_customers['monthly_profit'].mean():,.2f}")

# Card type analysis
print(f"\n💳 Card Type Performance:")
card_analysis = df.groupby('card_type').agg({
    'monthly_profit': 'mean',
    'avg_monthly_spend': 'mean',
    'customer_id': 'count'
}).round(2)
card_analysis.columns = ['Avg_Monthly_Profit', 'Avg_Monthly_Spend', 'Customer_Count']
print(card_analysis.sort_values('Avg_Monthly_Profit', ascending=False))

# Risk indicators
high_risk = df[(df['utilization_rate'] > 0.8) | (df['late_payments_12m'] > 6)]
print(f"\n⚠️ Risk Analysis:")
print(f"  • High-risk customers: {len(high_risk):,} ({len(high_risk)/len(df)*100:.1f}%)")
print(f"  • Inactive customers: {len(df[~df['is_active']]):,} ({len(df[~df['is_active']])/len(df)*100:.1f}%)")
print(f"  • Average utilization rate: {df['utilization_rate'].mean():.3f}")

# Revenue streams
print(f"\n💰 Revenue Analysis:")
total_interchange = df['interchange_revenue'].sum()
total_interest = df['interest_revenue'].sum()
total_fees = df['annual_fee'].sum() / 12  # Monthly equivalent
total_revenue = total_interchange + total_interest + total_fees

print(f"  • Monthly interchange revenue: ₹{total_interchange:,.2f} ({total_interchange/total_revenue*100:.1f}%)")
print(f"  • Monthly interest revenue: ₹{total_interest:,.2f} ({total_interest/total_revenue*100:.1f}%)")
print(f"  • Monthly fee revenue: ₹{total_fees:,.2f} ({total_fees/total_revenue*100:.1f}%)")
print(f"  • Total monthly revenue: ₹{total_revenue:,.2f}")

## 9. Save Data for CLV Analysis

In [ ]:
# Save to CSV file
filename = 'syntheticdata.csv'
df.to_csv(filename, index=False)

print(f"✅ Data saved to {filename}")
print(f"📁 File size: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"📊 Records: {len(df):,}")
print(f"📋 Features: {len(df.columns)}")

# Display file info
if os.path.exists(filename):
    file_size = os.path.getsize(filename) / 1024**2
    print(f"💾 Saved file size: {file_size:.2f} MB")

## 10. Export Options for Different Use Cases

In [ ]:
# Export sample datasets for different scenarios
print("📤 Creating specialized datasets...")

# High-value customers only
high_value = df[df['monthly_profit'] > df['monthly_profit'].quantile(0.8)]
high_value.to_csv('high_value_customers.csv', index=False)
print(f"✅ High-value customers: {len(high_value):,} records saved")

# Risk analysis dataset
risk_features = ['customer_id', 'age', 'annual_income', 'utilization_rate', 'payment_ratio', 
                'late_payments_12m', 'monthly_profit', 'is_active']
risk_df = df[risk_features].copy()
risk_df.to_csv('risk_analysis_data.csv', index=False)
print(f"✅ Risk analysis dataset: {len(risk_df)} records with {len(risk_features)} features saved")

# Sample for quick testing
sample_df = df.sample(n=1000, random_state=42)
sample_df.to_csv('sample_data.csv', index=False)
print(f"✅ Sample dataset: 1,000 records saved for quick testing")

print("\n📁 Available files:")
for file in ['syntheticdata.csv', 'high_value_customers.csv', 'risk_analysis_data.csv', 'sample_data.csv']:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024
        print(f"  • {file}: {size:.1f} KB")

## 11. Experiment with Different Scenarios

In [ ]:
# Experiment: Generate data for different customer segments
print("🧪 EXPERIMENTAL SCENARIOS")
print("=" * 50)

scenarios = {
    'Young Professionals': {
        'num_records': 2000,
        'min_age': 22,
        'max_age': 35,
        'min_income': 400000,
        'max_income': 1200000,
        'churn_rate': 0.20
    },
    'High Net Worth': {
        'num_records': 1000,
        'min_age': 35,
        'max_age': 60,
        'min_income': 1500000,
        'max_income': 5000000,
        'churn_rate': 0.10
    },
    'Risk Portfolio': {
        'num_records': 1500,
        'min_age': 18,
        'max_age': 75,
        'min_income': 200000,
        'max_income': 800000,
        'churn_rate': 0.35
    }
}

# Generate each scenario (uncomment to run)
# for scenario_name, params in scenarios.items():
#     print(f"\nGenerating {scenario_name} scenario...")
#     scenario_df = generate_synthetic_data(**params)
#     filename = f"{scenario_name.lower().replace(' ', '_')}_scenario.csv"
#     scenario_df.to_csv(filename, index=False)
#     print(f"✅ Saved {len(scenario_df):,} records to {filename}")

print("\n💡 To generate scenario data, uncomment the loop above and run the cell!")
print("\n🎯 Try modifying the CONFIG dictionary at the top and re-running the generation!")

## 12. Next Steps

In [ ]:
print("🎯 NEXT STEPS")
print("=" * 50)
print("1. ✅ Data generation completed")
print("2. 📊 Proceed to CLV Analysis notebook")
print("3. 🔍 Explore customer segmentation")
print("4. 💰 Calculate lifetime value metrics")
print("5. 📈 Generate business insights")

print("\n🔗 Links:")
print("• CLV Analysis Notebook: clv_analysis_demo.ipynb")
print("• GitHub Repository: https://github.com/VinayaSharada/FinancialAnalyticsCourse")

print("\n📚 What you learned:")
print("• How to generate realistic synthetic financial data")
print("• Data validation and quality checking techniques")
print("• Business metrics calculation for credit cards")
print("• Revenue and cost modeling for CLV analysis")

print("\n🏦 Banking Applications:")
print("• Customer acquisition cost optimization")
print("• Risk-based pricing strategies")
print("• Portfolio management and diversification")
print("• Regulatory reporting and compliance")